# 01 — Daily prices for the stock universe, SPY and the Treasury-bill yield

**Reads:** Yahoo Finance (only when `REFETCH = True`). No API key is needed.

**Writes:** `data/raw/prices_yahoo.csv`, `data/raw/fetch_log.json`, `data/processed/price_coverage.csv`.

| Column | What it is |
|---|---|
| `symbol` | ticker; 96 stocks, plus `SPY` (S&P 500 ETF) and `^IRX` (13-week Treasury-bill yield, % a year) |
| `date` | trading day |
| `open`, `high`, `low`, `close` | daily prices, split-adjusted |
| `volume` | shares traded |
| `adjusted` | close adjusted for splits **and dividends**; returns computed from it are total returns |

The universe is the list in the class script (`reference/original_XGBoostStockPrediction.R`). That list says
"100 large U.S. companies" but contains 96 tickers. One of them, `BK` (Bank of New York Mellon), no longer
returns a price history on Yahoo Finance; the company's full history is listed under `BNY`, which is used here.

The file in the repository was downloaded on 13 November 2026 from Yahoo Finance's chart endpoint. Setting
`REFETCH = True` downloads it again with `yfinance`, which reads the same endpoint, and overwrites the file.
The history starts in 2010 because the improved model trains on a longer window than the class script,
which starts in 2022.

In [1]:
from pathlib import Path
import json, datetime as dt
import pandas as pd

ROOT = Path.cwd().parent           # notebooks run from code/
RAW = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
RAW.mkdir(parents=True, exist_ok=True)
PROC.mkdir(parents=True, exist_ok=True)

REFETCH = False                    # True: download again from Yahoo Finance and overwrite prices_yahoo.csv
START = "2010-01-01"

STOCKS = [
    # Technology
    "AAPL", "MSFT", "NVDA", "GOOG", "META", "AMZN", "AVGO", "ORCL", "ADBE", "CRM",
    "CSCO", "AMD", "INTC", "QCOM", "TXN", "IBM", "NOW", "INTU", "AMAT", "MU",
    # Financials (BK in the class script; Yahoo Finance now lists the company as BNY)
    "JPM", "BAC", "WFC", "GS", "MS", "BLK", "C", "SCHW", "SPGI", "AXP",
    "USB", "PNC", "TFC", "BNY", "COF",
    # Healthcare
    "UNH", "JNJ", "LLY", "PFE", "MRK", "ABBV", "TMO", "DHR", "ABT", "BMY",
    "AMGN", "GILD", "CVS", "ISRG", "MDT",
    # Consumer discretionary
    "TSLA", "HD", "MCD", "NKE", "SBUX", "LOW", "TJX", "BKNG", "CMG", "ORLY",
    # Consumer staples
    "WMT", "COST", "PG", "KO", "PEP", "PM", "MO", "CL", "EL",
    # Industrials
    "CAT", "GE", "HON", "UPS", "BA", "RTX", "DE", "LMT", "MMM", "ETN",
    # Energy
    "XOM", "CVX", "COP", "SLB", "EOG",
    # Communication / media
    "NFLX", "DIS", "CMCSA", "TMUS", "VZ",
    # Real estate / utilities / materials
    "AMT", "PLD", "NEE", "DUK", "SO", "LIN", "APD",
]
OTHER = ["SPY", "^IRX"]
len(STOCKS)

96

## Download (only when `REFETCH = True`)

`yfinance` returns a two-level column index in recent versions; the first level is kept. `auto_adjust=False`
keeps both the plain close and the dividend-adjusted close, which is the column the class script calls `adjusted`.

In [2]:
if REFETCH:
    import yfinance as yf

    frames = []
    for sym in STOCKS + OTHER:
        px = yf.download(sym, start=START, auto_adjust=False, progress=False)
        if isinstance(px.columns, pd.MultiIndex):
            px.columns = px.columns.get_level_values(0)
        px = px.reset_index().rename(columns={"Date": "date", "Open": "open", "High": "high", "Low": "low",
                                              "Close": "close", "Volume": "volume", "Adj Close": "adjusted"})
        px["date"] = pd.to_datetime(px["date"]).dt.tz_localize(None).dt.normalize()
        px["symbol"] = sym
        frames.append(px[["symbol", "date", "open", "high", "low", "close", "volume", "adjusted"]]
                      .dropna(subset=["close", "adjusted"]))
    prices = pd.concat(frames, ignore_index=True)
    prices.to_csv(RAW / "prices_yahoo.csv", index=False)
    log = {"run_date": dt.date.today().isoformat(), "method": "yfinance, one ticker at a time, auto_adjust=False",
           "source": "Yahoo Finance", "start": START, "symbols": len(STOCKS + OTHER), "rows": int(len(prices)),
           "first": prices["date"].min().date().isoformat(), "last": prices["date"].max().date().isoformat()}
    json.dump(log, open(RAW / "fetch_log.json", "w"), indent=2)

prices = pd.read_csv(RAW / "prices_yahoo.csv", parse_dates=["date"])
print(json.dumps(json.load(open(RAW / "fetch_log.json")), indent=2))

{
  "run_date": "2026-11-13",
  "method": "Yahoo Finance chart endpoint (query1.finance.yahoo.com/v8/finance/chart/<ticker>, interval=1d, from 2010-01-01), one ticker at a time; notebook 01 re-downloads with yfinance when REFETCH = True",
  "source": "Yahoo Finance",
  "start": "2010-01-01",
  "symbols": 98,
  "rows": 413611,
  "first": "2010-01-04",
  "last": "2026-11-12",
  "note": "BK (Bank of New York Mellon) returned no price history; the company's history is listed under BNY, which is used instead. ^IRX has two fewer trading days than SPY."
}


## Checks

Every ticker in the list must be in the file, with no missing prices and no trading days skipped relative to SPY.
Stocks that listed after 2010 start later; the models only use a stock once it has a year of history.

In [3]:
missing = sorted(set(STOCKS + OTHER) - set(prices["symbol"]))
assert not missing, f"tickers missing from the file: {missing}"
assert prices[["open", "high", "low", "close", "adjusted"]].notna().all().all()

spy_days = set(prices.loc[prices["symbol"] == "SPY", "date"])
coverage = (prices.groupby("symbol")
            .agg(first=("date", "min"), last=("date", "max"), rows=("date", "size"))
            .reset_index())
coverage["days_missing_vs_SPY"] = [
    len({d for d in spy_days if d >= f} - set(prices.loc[prices["symbol"] == s, "date"]))
    for s, f in zip(coverage["symbol"], coverage["first"])]
coverage.to_csv(PROC / "price_coverage.csv", index=False)

print(f"{len(prices):,} rows, {prices['symbol'].nunique()} symbols, "
      f"{prices['date'].min().date()} to {prices['date'].max().date()}")
print("start after 2010-01-04:")
print(coverage[coverage["first"] > "2010-01-04"].to_string(index=False))
print("days missing relative to SPY:", coverage.loc[coverage["days_missing_vs_SPY"] > 0, ["symbol", "days_missing_vs_SPY"]].values.tolist())

413,611 rows, 98 symbols, 2010-01-04 to 2026-11-12
start after 2010-01-04:
symbol      first       last  rows  days_missing_vs_SPY
  ABBV 2013-01-02 2026-11-12  3488                    0
  META 2012-05-18 2026-11-12  3643                    0
   NOW 2012-06-29 2026-11-12  3614                    0
  TSLA 2010-06-29 2026-11-12  4120                    0
days missing relative to SPY: [['^IRX', 2]]
